# NullVector + LangGraph: Search-Based Research Agent

Unlike a simple linear QA pipeline, deep research often requires an iterative search approach. 

This notebook demonstrates how to create a **LangGraph ReAct Agent** that uses the NullVector `RetrievalService` as a tool. The LLM can autonomously decide to issue one or multiple search queries against the ingested document corpus to gather sufficient grounded evidence before formulating a final answer.

In [1]:
# 1. Install required dependencies
# !pip install -q langgraph langchain-openai langchain-core pydantic pymupdf pypdf litellm httpx

In [2]:
import os
import sys

# Ensure the nullvector package is in the path (assuming notebook is at project root)
sys.path.append(os.path.abspath("."))

### 1. Setup Phase: Generate Mock PDF & Authenticate
We create our sample research paper and configure OpenRouter for both NullVector's internal processes and the LangChain agent.

In [ ]:
# def create_mock_pdf(filename="sample_research.pdf"):
#     doc = fitz.open()

#     page1 = doc.new_page()
#     page1.insert_text((50, 50), "1. Introduction", fontsize=16)
#     page1.insert_text((50, 80), "Quantum computing leverages superposition and entanglement to perform complex calculations.\n"
#                                 "This research analyzes the decoherence rates of superconducting qubits.\n", fontsize=12)

#     page2 = doc.new_page()
#     page2.insert_text((50, 50), "2. Methodology", fontsize=16)
#     page2.insert_text((50, 80), "We utilize a cryogenic dilution refrigerator stabilized at 15 milliKelvin.\n"
#                                 "The readout is performed using a dispersive microwave resonator coupled to the qubit.\n", fontsize=12)

#     doc.save(filename)
#     doc.close()
#     return os.path.abspath(filename)

SOURCE_PDF = "/home/pruthvi/projects/NullVector/cookbook/903000608.pdf"

# --- OpenRouter Configuration ---
OPENROUTER_KEY = "REDACTED_OPENROUTER_KEY"
MODEL_NAME = "openrouter/google/gemini-3.1-flash-lite-preview"
API_BASE = "https://openrouter.ai/api/v1"

from nullvector.llm import GatewayConfig, GatewayService
from nullvector.llm.adapters import LiteLLMAdapter

llm_config = GatewayConfig(
    default_model=MODEL_NAME,
)
gateway = GatewayService(llm_config, provider_adapter=LiteLLMAdapter())

### 2. Standard ETL Ingestion
We run the linear processing steps to parse, build the hierarchy, and extract semantic blocks into a queryable corpus.

In [ ]:
from nullvector.domain.models import (
    AcquisitionRequest,
    AcquisitionSettings,
    TreeBuildRequest,
    TreeSettings,
)
from nullvector.ingest import acquire_document
from nullvector.retrieval import RetrievalCorpusBuilder
from nullvector.tree import build_tree


def process_document_to_corpus(source_path: str) -> str:
    artifact_root_path = os.path.abspath("artifacts/search_agent_ingest")

    # 1. Acquire
    req = AcquisitionRequest(
        source_path=os.path.abspath(source_path),
        acquisition_run_id="run_search_001",
        artifact_root=artifact_root_path,
        provider_identity="native_pymupdf",
        settings=AcquisitionSettings(detect_tables=True),
    )
    manifest = acquire_document(req)
    manifest_path = os.path.join(
        artifact_root_path, "run_search_001", manifest.document_id, "manifest.json"
    )

    # 2. Build Tree
    tree_req = TreeBuildRequest(
        acquisition_manifest_path=manifest_path,
        tree_run_id="tree_search_001",
        summarize=True,
        settings=TreeSettings(),
    )
    tree_manifest = build_tree(tree_req, gateway=gateway)
    tree_path = f"{tree_manifest.artifact_root}/manifest.json"

    # 3. Build Corpus
    builder = RetrievalCorpusBuilder()
    corpus_root_path = os.path.abspath(f"artifacts/search_corpus/{manifest.document_id}")
    corpus_manifest = builder.build(
        acquisition_manifest_path=manifest_path,
        tree_manifest_path=tree_path,
        artifact_root=corpus_root_path,
    )
    return corpus_manifest.corpus_path


CORPUS_PATH = process_document_to_corpus(SOURCE_PDF)
print(f"\nCorpus ready at: {CORPUS_PATH}")

### 3. Build the LangChain Search Tool
We wrap `nullvector.retrieval.RetrievalService` into a `@tool`. This allows the LangGraph agent to pass dynamic queries and get grounded excerpts back.

In [ ]:
from langchain_core.tools import tool

from nullvector.retrieval import (
    QueryPlanner,
    RetrievalRanker,
    RetrievalService,
    load_retrieval_corpus,
)


@tool
def search_document(query: str) -> str:
    """Search the ingested research document for specific facts, metrics, or methodology details."""
    corpus = load_retrieval_corpus(CORPUS_PATH)

    planner = QueryPlanner()
    ranker = RetrievalRanker()
    retrieval_service = RetrievalService(planner, ranker)

    hits = retrieval_service.search(corpus=corpus, query=query, limit=3)

    if not hits:
        return "No relevant information found in the document for this query."

    results = []
    for i, hit in enumerate(hits):
        page = hit.unit.page_span.start_page + 1  # Convert to 1-based indexing for readability
        title = hit.unit.title or "Section"
        text = hit.unit.text or ""
        # Clean up text for the agent
        text_snippet = " ".join(text.split())[:500]  # Provide a solid 500 char snippet

        results.append(f"Hit {i + 1} [Page {page} - {title}]: {text_snippet}")

    return "\n\n".join(results)

### 4. Create the LangGraph Agent
Using `create_react_agent`, we bind the Search tool to our LLM. The agent will read your prompt, realize it needs document context, call `search_document`, read the results, and formulate a final answer.

In [ ]:
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent

# LangChain's OpenAI client is compatible with OpenRouter when provided the correct base_url
llm = ChatOpenAI(api_key=OPENROUTER_KEY, base_url=API_BASE, model=MODEL_NAME, temperature=0)

tools = [search_document]

# Create the LangGraph state machine agent
search_agent = create_react_agent(llm, tools)

### 5. Execute Search Agent
Watch the agent's internal thought process as it invokes tools and streams the final synthesis.

In [ ]:
def ask_agent(question: str):
    print(f"\nUser Question: {question}\n")
    print("--- Agent Execution Trace ---")

    # Initialize state with the user message
    inputs = {"messages": [("user", question)]}

    # Stream the graph execution to observe tool calls
    for event in search_agent.stream(inputs, stream_mode="values"):
        last_message = event["messages"][-1]

        # Check if it's a tool call (Action)
        if getattr(last_message, "tool_calls", None):
            for tool_call in last_message.tool_calls:
                print(
                    f"🔧 Agent decided to act: Calling '{tool_call['name']}' with args: {tool_call['args']}"
                )

        # Check if it's a tool response (Observation)
        elif last_message.type == "tool":
            print(f"📄 Observation from '{last_message.name}':\n{last_message.content}\n")

    print("\n" + "=" * 50)
    print("FINAL ANSWER")
    print("=" * 50)
    print(event["messages"][-1].content)


# Ask a specific question that requires checking the methodology section
ask_agent(
    "What is the specific temperature the refrigerator is stabilized at, and what kind of resonator is used for readout?"
)